<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/07-RAG_Improve_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Improving RAG with Metadata-Enriched Chunking

Raw text chunks retrieved from a vector store often lack context — they answer *what* is stored but not *why* it matters. In this notebook, we enrich each chunk with **auto-generated metadata** (summaries, questions, keywords) during the ingestion pipeline, then evaluate whether this richer representation improves retrieval quality.

## What You'll Learn
- How to use LlamaIndex's `QuestionsAnsweredExtractor`, `SummaryExtractor`, and `KeywordExtractor` in an ingestion pipeline
- How metadata enrichment affects retrieval Hit Rate and MRR
- How to compare a metadata-enriched index against a plain-text baseline

## Prerequisites
- Basic familiarity with Python and RAG concepts
- API keys for OpenAI and Google Gemini (added to Colab Secrets)

# Install Packages and Setup Variables


In [ ]:
!pip install -q llama-index==0.14.20 openai==2.21.0 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.3 \
                llama-index-llms-google-genai==0.8.7 cohere==5.20.5 ipywidgets==7.7.1 jedi==0.19.2

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

nest_asyncio.apply()

# Load a Model


In [ ]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "minimal"})

In [ ]:
# Gemini (Free Tier) — RPM quota errors might occur with many LLM calls.
# Uncomment to use Gemini instead of GPT:
# from llama_index.llms.google_genai import GoogleGenAI
# llm = GoogleGenAI(model="gemini-2.5-flash", temperature=0.3)

# Create a Vector Store


In [ ]:
import chromadb

# Create client and a new collection.
# chromadb.EphemeralClient saves data in-memory.
chroma_client = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = chroma_client.create_collection("mini-llama-articles")

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Load the Dataset (CSV)


## Download


The dataset includes several articles from the TowardsAI blog with an in-depth explanation of the LLaMA2 model.


In [ ]:
!curl -o ./mini-llama-articles.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

## Read File


In [ ]:
import csv

rows = []

# Load the CSV file
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            # Skip header row
            continue
        rows.append(row)

print(f"Loaded {len(rows)} rows")

# Convert to Document Objects


In [ ]:
from llama_index.core import Document

# Convert the chunks to Document objects so the LlamaIndex framework can process them.
documents = [
    Document(
        text=row[1], metadata={"title": row[0], "url": row[2], "source_name": row[3]}
    )
    for row in rows
]

print(f"Created {len(documents)} documents")

# Transforming

Here we build a richer ingestion pipeline. In addition to chunking with `TokenTextSplitter`, we attach three types of auto-generated metadata to each chunk:

- **`QuestionsAnsweredExtractor`** — generates 3 questions the chunk can answer
- **`SummaryExtractor`** — generates a summary of the previous and current chunk
- **`KeywordExtractor`** — extracts 10 keywords from the chunk


In [ ]:
from llama_index.core.node_parser import TokenTextSplitter

# Define the splitter: 512-token chunks with 128-token overlap.
text_splitter = TokenTextSplitter(separator=" ", chunk_size=512, chunk_overlap=128)

In [ ]:
from llama_index.core.extractors import (
    SummaryExtractor,
    QuestionsAnsweredExtractor,
    KeywordExtractor,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

# Build the pipeline: chunk → enrich with metadata → embed → store.
pipeline = IngestionPipeline(
    transformations=[
        text_splitter,
        QuestionsAnsweredExtractor(questions=3, llm=llm),
        SummaryExtractor(summaries=["prev", "self"], llm=llm),
        KeywordExtractor(keywords=10, llm=llm),
        OpenAIEmbedding(),
    ],
    vector_store=vector_store,
)

# Run the transformation pipeline.
nodes = pipeline.run(documents=documents, show_progress=True)

In [ ]:
print(f"Ingested {len(nodes)} nodes")

In [ ]:
# Compress the vector store directory to a zip file so you can download and reuse it later.
!zip -r vectorstore.zip mini-llama-articles

# Load Indexes


If you have already uploaded the zip file for the vector store checkpoint, uncomment the cell below to extract it. You can then load the index from local storage without re-running the pipeline.


In [ ]:
# !unzip vectorstore.zip

In [ ]:
# Load the vector store from local storage.
db = chromadb.PersistentClient(path="./mini-llama-articles")
chroma_collection = db.get_or_create_collection("mini-llama-articles")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
from llama_index.core import VectorStoreIndex

# Create the index from the vector store.
index = VectorStoreIndex.from_vector_store(vector_store)

# Query Dataset


In [ ]:
# Define a query engine that retrieves related chunks and formulates a final answer.
query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

res = query_engine.query("How many parameters does the LLaMA 2 model have?")

In [ ]:
print(res.response)

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text[:200])
    print("Score\t", src.score)
    print("-_" * 20)

### Trying a Different Query


In [ ]:
res = query_engine.query("Did GQA help LLaMA performance?")

In [ ]:
print(res.response)

In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text[:200])
    print("Score\t", src.score)
    print("-_" * 20)

From the articles:

> [...] The 7 billion model of Llama2 has sufficient NLU (Natural Language Understanding) to create output based on a particular format [...]


# Baseline: No Metadata


Now let's evaluate the query engine independently of the generated metadata — without keyword extraction, summaries, or questions. This gives us a plain-text baseline to compare against.


In [ ]:
from llama_index.core import Document

documents_no_meta = [Document(text=row[1]) for row in rows]

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.ingestion import IngestionPipeline

pipeline_no_meta = IngestionPipeline(
    transformations=[
        text_splitter,
        OpenAIEmbedding(),
    ]
)

nodes_no_meta = pipeline_no_meta.run(documents=documents_no_meta, show_progress=True)
print(f"Baseline nodes (no metadata): {len(nodes_no_meta)}")

In [ ]:
index_no_metadata = VectorStoreIndex(nodes=nodes_no_meta)
query_engine_no_metadata = index_no_metadata.as_query_engine(llm=llm)

In [ ]:
res = query_engine_no_metadata.query("Did GQA help LLaMA performance?")

In [ ]:
print(res.response)

In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text[:200])
    print("Score\t", src.score)
    print("-_" * 20)

# Evaluate

Now we generate an evaluation dataset and compare the metadata-enriched index against the baseline using Hit Rate and MRR.


In [ ]:
from llama_index.core.evaluation import generate_question_context_pairs

# Generate one question per node. These questions are used to assess whether
# the retriever can identify and return the correct chunk when queried.
rag_eval_dataset = generate_question_context_pairs(
    nodes, llm=llm, num_questions_per_chunk=1
)

# Save the evaluation dataset as a JSON file for later use.
rag_eval_dataset.save_json("./rag_eval_dataset.json")

If you have already uploaded the generated question JSON file, uncomment the cell below to load it directly — skipping question generation entirely.


In [ ]:
# from llama_index.finetuning.embeddings.common import (
#     EmbeddingQAFinetuneDataset,
# )
# rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json(
#     "./rag_eval_dataset.json"
# )

In [ ]:
import pandas as pd


def display_results_retriever(name, eval_results):
    """Display retrieval evaluation results as a DataFrame."""
    metric_dicts = [result.metric_vals_dict for result in eval_results]
    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    return pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

In [ ]:
from llama_index.core.evaluation import RetrieverEvaluator

# Evaluate the retriever with different top_k values.
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(similarity_top_k=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

In [ ]:
from llama_index.core.evaluation import (
    RelevancyEvaluator,
    FaithfulnessEvaluator,
    BatchEvalRunner,
)
from llama_index.llms.openai import OpenAI

for i in [2, 4, 6, 8, 10]:
    query_engine = index.as_query_engine(similarity_top_k=i, llm=llm)

    # Use GPT-5 as the evaluation judge
    llm_gpt5 = OpenAI(model="gpt-5", additional_kwargs={"reasoning_effort": "minimal"})

    faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_gpt5)
    relevancy_evaluator = RelevancyEvaluator(llm=llm_gpt5)

    queries = list(rag_eval_dataset.queries.values())
    batch_eval_queries = queries[:20]

    runner = BatchEvalRunner(
        {"faithfulness": faithfulness_evaluator, "relevancy": relevancy_evaluator},
        workers=8,
    )
    eval_results = await runner.aevaluate_queries(
        query_engine, queries=batch_eval_queries
    )

    faithfulness_score = sum(
        result.passing for result in eval_results["faithfulness"]
    ) / len(eval_results["faithfulness"])
    print(f"top_{i} faithfulness_score: {faithfulness_score:.2f}")

    relevancy_score = sum(result.passing for result in eval_results["relevancy"]) / len(
        eval_results["relevancy"]
    )
    print(f"top_{i} relevancy_score: {relevancy_score:.2f}")
    print("-_" * 10)

## Optional: Compare Metadata vs. Baseline Retrieval

This section runs a side-by-side Hit Rate and MRR comparison between the metadata-enriched index and the plain-text baseline — for a fixed set of evaluation questions.


In [ ]:
import asyncio

async def evaluate_retriever(retriever, dataset, name):
    """Run retriever evaluation and return a summary row."""
    evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    results = await evaluator.aevaluate_dataset(dataset)
    metric_dicts = [r.metric_vals_dict for r in results]
    df = pd.DataFrame(metric_dicts)
    return {
        "Retriever": name,
        "Hit Rate": round(df["hit_rate"].mean(), 3),
        "MRR": round(df["mrr"].mean(), 3),
    }


async def run_comparison(top_k=5):
    """Compare metadata-enriched index vs. plain baseline at a given top_k."""
    rows_out = []

    r1 = index.as_retriever(similarity_top_k=top_k)
    rows_out.append(await evaluate_retriever(r1, rag_eval_dataset, f"With Metadata (top_{top_k})"))

    r2 = index_no_metadata.as_retriever(similarity_top_k=top_k)
    rows_out.append(await evaluate_retriever(r2, rag_eval_dataset, f"No Metadata (top_{top_k})"))

    return pd.DataFrame(rows_out)


# Run the comparison for top_k = 5
comparison_df = await run_comparison(top_k=5)
print(comparison_df.to_string(index=False))

### Interactive Top-K Comparison

Use the slider below to compare both retrievers at different `top_k` values.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

topk_widget = widgets.IntSlider(
    value=5, min=1, max=10, step=1, description="top_k:"
)
run_btn = widgets.Button(description="Compare", button_style="primary")
out = widgets.Output()


def on_compare(b):
    with out:
        clear_output()
        print(f"Comparing at top_k={topk_widget.value}...")
        result = asyncio.get_event_loop().run_until_complete(
            run_comparison(top_k=topk_widget.value)
        )
        print(result.to_string(index=False))


run_btn.on_click(on_compare)
display(widgets.VBox([topk_widget, run_btn, out]))